# Ćwiczenia 1 — Środowisko, dane i kształt p ≫ n (wersja Python)
**Dane demo: mikrobiom jelitowy w cukrzycy typu 2 (QinJ_2012)**

## 1. Wczytanie kohorty demo

Wersja Python pracuje na danych wyeksportowanych z R do CSV (patrz wersja R, sekcja 2). `X` to próbki × taksony, `y` to etykieta grupy, `meta` to metadane kliniczne. Plik etykiet ma jedną kolumnę `grupa`, którą spłaszczamy do serii.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# X: macierz cech (próbki × taksony), y: etykieta grupy, meta: metadane kliniczne
X = pd.read_csv("dane/t2d_features.csv", index_col=0)
y = pd.read_csv("dane/t2d_labels.csv").squeeze()
meta = pd.read_csv("dane/t2d_metadata.csv", index_col=0)

# klasa referencyjna = control, klasa pozytywna = T2D
y = pd.Series(pd.Categorical(y, categories=["control", "T2D"]))

print(X.shape)
print(y.value_counts())

Dane mają kształt `p ≫ n`: cech (taksonów) jest znacznie więcej niż próbek. To pierwsza i najważniejsza cecha problemu — wraca w każdej kolejnej sekcji kursu.

> **✏️ Zadanie 1 — Kształt danych.** Odczytać kodem (nie „na oko"):
>
> 1. Ile jest próbek, a ile taksonów? Który wymiar jest większy?
> 2. Jaki jest rozkład klas? Czy klasy są zbalansowane?
> 3. Jakiego typu są wartości cech i w jakim mieszczą się zakresie?
> 4. Jaki odsetek macierzy cech to zera? Co to mówi o danych?

In [ ]:
# TODO: odczytać wymiary, rozkład klas, typ i zakres wartości oraz odsetek zer

## 2. Zgodność wersji

Od tego spotkania obie wersje operują na tych samych danych. Potwierdzamy, że CSV wczytany w Pythonie zgadza się z tym, co wyeksportowała wersja R.

> **✏️ Zadanie 2 — Zgodność wersji.** Porównać wymiary macierzy i rozkład klas z wartościami odczytanymi w wersji R (`lab_01.Rmd`). Powinny być identyczne — to samo źródło danych.

In [ ]:
# TODO: wypisać X.shape oraz y.value_counts() i porównać z wersją R

## 3. Eksploracja cech

Po otwarciu danych warto je najpierw poznać. Wartości w `X` to **liczebności względne** (procenty profilu taksonomicznego), więc suma w każdej próbce jest w przybliżeniu stała. Rozkład pojedynczego taksonu jest zwykle silnie skośny: kilka taksonów dominuje profil, większość występuje śladowo lub wcale.

> **✏️ Zadanie 3 — Dominujące taksony i kompozycyjność.** Zbadać strukturę macierzy cech:
>
> 1. Policzyć sumę każdego wiersza `X` (`X.sum(axis=1)`). Jaką wartość przyjmuje i co z tego wynika o naturze danych?
> 2. Wyznaczyć 5 taksonów o najwyższej średniej liczebności (`X.mean(axis=0)`). Jaki udział profilu zajmuje sam najliczniejszy takson?
> 3. Narysować histogram liczebności najliczniejszego taksonu. Czy rozkład jest symetryczny, czy skośny? (zapowiedź transformacji CLR ze spotkania 3)

In [ ]:
# TODO: policzyć sumy wierszy, 5 najliczniejszych taksonów i histogram najliczniejszego

## 4. Metadane kohorty i zmienne zakłócające

Eksploracja to także pytanie, czy grupy `control` i `T2D` różnią się czymś **poza mikrobiomem**. Jeśli różnią się wiekiem, płcią czy głębokością sekwencjonowania, model może nauczyć się sygnału związanego z tymi zmiennymi, a nie z chorobą. Takie zmienne nazywamy **zakłócającymi** (confounders).

> **✏️ Zadanie 4 — Metadane i zmienne zakłócające.** Na podstawie `meta`:
>
> 1. Które zmienne są stałe w całej kohorcie (ta sama wartość u wszystkich próbek)? Co z nich wynika dla modelu?
> 2. Porównać wiek (`age`) między grupami: mediany i test Manna-Whitneya (`scipy.stats.mannwhitneyu`). Czy grupy są podobne?
> 3. Porównać płeć (`gender`) między grupami: `pd.crosstab` i test chi-kwadrat (`scipy.stats.chi2_contingency`).
> 4. Porównać głębokość sekwencjonowania (`number_reads`) między grupami.
> 5. Która z badanych zmiennych **nie** różni grup istotnie?
> 6. Jednym–dwoma zdaniami: dlaczego różnica wieku i płci między grupami jest problemem przy budowie klasyfikatora na mikrobiomie?

In [ ]:
# TODO: wskazać zmienne stałe; porównać age, gender, number_reads i BMI między grupami

## 5. Konsekwencje kształtu `p ≫ n` i wersje środowiska

Wersje pakietów zapisujemy od początku — to podstawa reprodukowalności. W wersji Python służy do tego `pip freeze` (z terminala) lub zapis wersji kluczowych bibliotek.

In [ ]:
import sys
print("Python", sys.version.split()[0])
for pkg in (pd, np):
    print(pkg.__name__, pkg.__version__)

> **✏️ Zadanie 5 — Konsekwencje kształtu `p ≫ n`.** Policzyć kodem:
>
> 1. Stosunek liczby cech do liczby próbek (`X.shape[1] / X.shape[0]`).
> 2. Ile taksonów ma zerową lub bliską zeru wariancję (`X.var(axis=0)`).
> 3. Ile taksonów występuje w mniej niż 10% próbek (`(X > 0).mean(axis=0)`).
> 4. Jednym zdaniem: dlaczego regresji logistycznej (właściwej dla binarnej etykiety) nie da się jednoznacznie dopasować na wszystkich cechach naraz, gdy `p > n`.

In [ ]:
# TODO: policzyć stosunek p/n, liczbę cech o zerowej wariancji i liczbę cech rzadkich

> **★ Zadanie rozszerzające.** Spróbować dopasować regresję logistyczną wiążącą etykietę ze wszystkimi taksonami jednocześnie. Sprawdzić, dla ilu współczynników procedura nie zwraca wartości, oraz policzyć odsetek trafnych klasyfikacji na tym samym zbiorze, który posłużył do dopasowania modelu. Rozstrzygnąć, czy taki wynik świadczy o jakości modelu, czy wynika z relacji `p > n`.